In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from pathlib import Path
from pprint import pprint
from collections import Counter
from typing import Dict, List
import json

from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_evaluator import InputSample
from presidio_evaluator.evaluation import SpanEvaluator, ModelError, Plotter
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import BaseModel
from presidio_evaluator.span_to_tag import span_to_tag

from hanlp_engine import HanLPNlpEngine, HanLPRecognizer
import hanlp

%reload_ext autoreload
%autoreload 2

dataset_name = "mixed_cn_en_500.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd(), "data", dataset_name), token_model_version="zh_core_web_sm")
print(len(dataset))

def get_entity_counts(dataset: List[InputSample]) -> Dict:
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print("\nMin and max number of tokens in dataset: "
      f"Min: {min([len(sample.tokens) for sample in dataset])}, "
      f"Max: {max([len(sample.tokens) for sample in dataset])}")

print(f"Min and max sentence length in dataset: "
      f"Min: {min([len(sample.full_text) for sample in dataset])}, "
      f"Max: {max([len(sample.full_text) for sample in dataset])}")


tokenizing input:   0%|          | 0/500 [00:00<?, ?it/s]

loading model zh_core_web_sm


tokenizing input: 100%|██████████| 500/500 [00:07<00:00, 63.08it/s]

500
Count per entity:
[('O', 31462), ('STREET_ADDRESS', 2635), ('PERSON', 1563),
 ('EMAIL_ADDRESS', 1190), ('AGE', 525), ('PHONE_NUMBER', 415), ('ID', 334),
 ('GENDER', 320), ('GPE', 149), ('ORGANIZATION', 111), ('DATE_TIME', 82),
 ('CREDIT_CARD', 45), ('DOMAIN_NAME', 43), ('TITLE', 43), ('IP_ADDRESS', 34),
 ('IBAN_CODE', 16), ('ZIP_CODE', 15), ('US_SSN', 15), ('NRP', 13)]

Min and max number of tokens in dataset: Min: 3, Max: 174
Min and max sentence length in dataset: Min: 9, Max: 626


In [3]:
sample = dataset[0]

for i, (token, tag) in enumerate(zip(sample.tokens, sample.tags)):
    print(f"{i:03d}\t{token}\t{tag}")


000	莫雅琴	PERSON
001	是	O
002	一	O
003	位	O
004	36	AGE
005	岁	AGE
006	的	O
007	女性	GENDER
008	交响	O
009	乐团	O
010	指挥	O
011	。	O
012	她	O
013	居住	O
014	在	O
015	浙江省	STREET_ADDRESS
016	杭州市	STREET_ADDRESS
017	西湖区	STREET_ADDRESS
018	文三路	STREET_ADDRESS
019	218号	STREET_ADDRESS
020	，	O
021	持有	O
022	身份	O
023	证号	O
024	330106198701211528	ID
025	。	O
026	她	O
027	的	O
028	电子	O
029	邮箱	O
030	是	O
031	moyaq	EMAIL_ADDRESS
032	in	EMAIL_ADDRESS
033	@	EMAIL_ADDRESS
034	163.com	EMAIL_ADDRESS
035	，	O
036	联系	O
037	电话	O
038	为	O
039	13857192468	PHONE_NUMBER
040	。	O
041	莫雅琴	PERSON
042	近期	O
043	出现	O
044	关节	O
045	疼痛	O
046	、	O
047	僵硬	O
048	和	O
049	活动	O
050	受限	O
051	的	O
052	症状	O
053	，	O
054	经	O
055	诊断	O
056	为	O
057	关节炎	O
058	。	O
059	她	O
060	的	O
061	主治	O
062	医生	O
063	张明远	PERSON
064	为	O
065	其	O
066	开具	O
067	了	O
068	阿司匹林	O
069	进行	O
070	治疗	O
071	。	O
072	财务	O
073	状况	O
074	方面	O
075	，	O
076	她	O
077	的	O
078	信用	O
079	评分	O
080	为	O
081	850分	O
082	，	O
083	年收入	O
084	为	O
085	921736	O
086	.	O
087	42	O
088	元	O
089	。	O
090	交易	O
091	记录	O
092	显示	O
0

In [4]:
def merge_results(*result_lists):
    merged = []
    seen = set()
    for results in result_lists:
        for result in results:
            key = (result.entity_type, result.start, result.end)
            if key in seen:
                continue
            seen.add(key)
            merged.append(result)
    return sorted(merged, key=lambda r: (r.start, r.end, r.entity_type))


class HanLPMixModelWrapper(BaseModel):
    def __init__(self, zh_analyzer, en_analyzer, entities_to_keep=None, labeling_scheme="IO", verbose=False):
        super().__init__(
            entities_to_keep=entities_to_keep,
            verbose=verbose,
            labeling_scheme=labeling_scheme,
        )
        self.zh_analyzer = zh_analyzer
        self.en_analyzer = en_analyzer
        self.name = "HanLP Mix Model Wrapper"

    def _predict_results(self, sample: InputSample):
        zh_results = self.zh_analyzer.analyze(text=sample.full_text, language="zh", entities=self.entities)
        en_results = self.en_analyzer.analyze(text=sample.full_text, language="en", entities=self.entities)
        return merge_results(zh_results, en_results)

    def predict(self, sample: InputSample, **kwargs) -> List[str]:
        results = self._predict_results(sample)
        starts, ends, tags, scores = [], [], [], []
        for res in results:
            starts.append(0 if res.start is None else res.start)
            ends.append(res.end)
            tags.append(res.entity_type)
            scores.append(res.score)
        return span_to_tag(
            scheme=self.labeling_scheme,
            text=sample.full_text,
            starts=starts,
            ends=ends,
            tags=tags,
            tokens=sample.tokens,
            scores=scores,
        )

    def batch_predict(self, dataset: List[InputSample], **kwargs) -> List[List[str]]:
        return [self.predict(sample, **kwargs) for sample in dataset]

    def to_log(self) -> Dict:
        data = super().to_log()
        data.update({"analyzer": "zh_hanlp_plus_en_spacy"})
        return data


In [5]:
# --- Chinese analyzer (HanLP) ---
from zh_pattern_recognizers import register_zh_pattern_recognizers



tok = hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)
ner = hanlp.load(hanlp.pretrained.ner.MSRA_NER_ELECTRA_SMALL_ZH)
hanlp_model = hanlp.pipeline().append(tok, output_key="tok").append(
    ner, input_key="tok", output_key="ner"
)

zh_nlp_engine = HanLPNlpEngine(hanlp_model)
zh_recognizer = HanLPRecognizer(zh_nlp_engine)

zh_registry = RecognizerRegistry(supported_languages=["zh"])
zh_registry.add_recognizer(zh_recognizer)
register_zh_pattern_recognizers(zh_registry)

zh_analyzer = AnalyzerEngine(
    nlp_engine=zh_nlp_engine,
    registry=zh_registry,
    supported_languages=["zh"],
)

# --- English analyzer (spaCy) ---
spacy_engine = NlpEngineProvider(
    nlp_configuration={
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
    }
).create_engine()

en_registry = RecognizerRegistry(supported_languages=["en"])
en_registry.load_predefined_recognizers(nlp_engine=spacy_engine)

en_analyzer = AnalyzerEngine(
    nlp_engine=spacy_engine,
    registry=en_registry,
    supported_languages=["en"],
)

pprint("Supported entities for Chinese:")
pprint(zh_analyzer.get_supported_entities("zh"), compact=True)

print("\nLoaded recognizers for Chinese:")
pprint([rec.name for rec in zh_analyzer.registry.get_recognizers("zh", all_fields=True)], compact=True)

pprint("\nSupported entities for English:")
pprint(en_analyzer.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint([rec.name for rec in en_analyzer.registry.get_recognizers("en", all_fields=True)], compact=True)


'Supported entities for Chinese:'
['AGE', 'PERSON', 'GENDER', 'ID', 'PHONE_NUMBER', 'LOCATION', 'EMAIL_ADDRESS']

Loaded recognizers for Chinese:
['HanLPRecognizer', 'EmailRecognizer', 'PhoneRecognizer', 'IdRecognizer',
 'GenderRecognizer']
'\nSupported entities for English:'
['US_SSN', 'CREDIT_CARD', 'AGE', 'PERSON', 'IP_ADDRESS', 'URL', 'PHONE_NUMBER',
 'NRP', 'LOCATION', 'MAC_ADDRESS', 'DATE_TIME', 'EMAIL_ADDRESS', 'EMAIL', 'ID',
 'US_PASSPORT', 'US_DRIVER_LICENSE', 'ORGANIZATION', 'US_ITIN', 'CRYPTO',
 'IBAN_CODE', 'UK_NHS', 'MEDICAL_LICENSE', 'US_BANK_NUMBER']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsLicenseRecognizer',
 'UsItinRecognizer', 'UsPassportRecognizer', 'UsSsnRecognizer', 'NhsRecognizer',
 'CryptoRecognizer', 'DateRecognizer', 'EmailRecognizer', 'IbanRecognizer',
 'IpRecognizer', 'MedicalLicenseRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'SpacyRecognizer']


In [6]:
from presidio_evaluator.models import PresidioAnalyzerWrapper

entities_mapping=PresidioAnalyzerWrapper.presidio_entities_map # default mapping
# Add titles and zip codes as we have recognizers for those
entities_mapping["GENDER"] = "GENDER"
# entities_mapping["STREET_ADDRESS"] = "LOCATION"
print("Entities mapping:")
pprint(entities_mapping)

dataset = SpanEvaluator.align_entity_types(
    dataset, 
    entities_mapping=entities_mapping, 
    allow_missing_mappings=True
)
new_entity_counts = get_entity_counts(dataset)
print("\nCount per entity after alignment:")
pprint(new_entity_counts.most_common(), compact=True)

dataset_entities = list(new_entity_counts.keys())

Entities mapping:
{'ADDRESS': 'LOCATION',
 'AGE': 'AGE',
 'BIRTHDAY': 'DATE_TIME',
 'CITY': 'LOCATION',
 'CREDIT_CARD': 'CREDIT_CARD',
 'CREDIT_CARD_NUMBER': 'CREDIT_CARD',
 'DATE': 'DATE_TIME',
 'DATE_OF_BIRTH': 'DATE_TIME',
 'DATE_TIME': 'DATE_TIME',
 'DOB': 'DATE_TIME',
 'DOMAIN': 'URL',
 'DOMAIN_NAME': 'URL',
 'EMAIL': 'EMAIL_ADDRESS',
 'EMAIL_ADDRESS': 'EMAIL_ADDRESS',
 'FACILITY': 'LOCATION',
 'FIRST_NAME': 'PERSON',
 'GENDER': 'GENDER',
 'GPE': 'LOCATION',
 'HCW': 'PERSON',
 'HOSP': 'ORGANIZATION',
 'HOSPITAL': 'ORGANIZATION',
 'IBAN': 'IBAN_CODE',
 'IBAN_CODE': 'IBAN_CODE',
 'ID': 'ID',
 'IP_ADDRESS': 'IP_ADDRESS',
 'LAST_NAME': 'PERSON',
 'LOC': 'LOCATION',
 'LOCATION': 'LOCATION',
 'NAME': 'PERSON',
 'NATIONALITY': 'NRP',
 'NORP': 'NRP',
 'NRP': 'NRP',
 'O': 'O',
 'ORG': 'ORGANIZATION',
 'ORGANIZATION': 'ORGANIZATION',
 'PATIENT': 'PERSON',
 'PATORG': 'ORGANIZATION',
 'PER': 'PERSON',
 'PERSON': 'PERSON',
 'PHONE': 'PHONE_NUMBER',
 'PHONE_NUMBER': 'PHONE_NUMBER',
 'PREFIX': '

In [7]:
sample = dataset[0]
text = sample.full_text

zh_results = zh_analyzer.analyze(text=text, language="zh", return_decision_process=True)
en_results = en_analyzer.analyze(text=text, language="en", return_decision_process=True)
combined_results = merge_results(zh_results, en_results)

print(text)
print("\nzh_results:")
print(zh_results)
print("\nen_results:")
print(en_results)
print("\ncombined_results:")
print(combined_results)


莫雅琴是一位36岁的女性交响乐团指挥。她居住在浙江省杭州市西湖区文三路218号，持有身份证号330106198701211528。她的电子邮箱是moyaqin@163.com，联系电话为13857192468。莫雅琴近期出现关节疼痛、僵硬和活动受限的症状，经诊断为关节炎。她的主治医生张明远为其开具了阿司匹林进行治疗。财务状况方面，她的信用评分为850分，年收入为921736.42元。交易记录显示有一笔'收入转账 REMI17121'的款项。

zh_results:
[type: ID, start: 46, end: 64, score: 0.95, type: PHONE_NUMBER, start: 93, end: 104, score: 0.9, type: AGE, start: 6, end: 9, score: 0.85, type: LOCATION, start: 23, end: 26, score: 0.85, type: LOCATION, start: 26, end: 29, score: 0.85, type: LOCATION, start: 29, end: 32, score: 0.85, type: LOCATION, start: 32, end: 39, score: 0.85, type: EMAIL_ADDRESS, start: 72, end: 87, score: 0.85, type: PERSON, start: 105, end: 108, score: 0.85, type: PERSON, start: 142, end: 145, score: 0.85, type: GENDER, start: 10, end: 12, score: 0.8, type: PERSON, start: 0, end: 3, score: 0.5]

en_results:
[type: EMAIL_ADDRESS, start: 65, end: 87, score: 1.0, type: URL, start: 80, end: 87, score: 0.5, type: PHONE_NUMBER, start: 93, end: 104, score: 0.4]

combined_results:
[type

In [8]:
for result in combined_results:
    print(f"Entity: {result.entity_type}, Text: {text[result.start:result.end]}, Span: ({result.start}, {result.end}), Score: {result.score}")


Entity: PERSON, Text: 莫雅琴, Span: (0, 3), Score: 0.5
Entity: AGE, Text: 36岁, Span: (6, 9), Score: 0.85
Entity: GENDER, Text: 女性, Span: (10, 12), Score: 0.8
Entity: LOCATION, Text: 浙江省, Span: (23, 26), Score: 0.85
Entity: LOCATION, Text: 杭州市, Span: (26, 29), Score: 0.85
Entity: LOCATION, Text: 西湖区, Span: (29, 32), Score: 0.85
Entity: LOCATION, Text: 文三路218号, Span: (32, 39), Score: 0.85
Entity: ID, Text: 330106198701211528, Span: (46, 64), Score: 0.95
Entity: EMAIL_ADDRESS, Text: 她的电子邮箱是moyaqin@163.com, Span: (65, 87), Score: 1.0
Entity: EMAIL_ADDRESS, Text: moyaqin@163.com, Span: (72, 87), Score: 0.85
Entity: URL, Text: 163.com, Span: (80, 87), Score: 0.5
Entity: PHONE_NUMBER, Text: 13857192468, Span: (93, 104), Score: 0.9
Entity: PERSON, Text: 莫雅琴, Span: (105, 108), Score: 0.85
Entity: PERSON, Text: 张明远, Span: (142, 145), Score: 0.85


In [9]:
dataset[2].full_text

'沈雅琴是一位59岁的女性营养师，现居住于江苏省南京市鼓楼区中山北路85号。您可以通过邮箱shenyaqin@163.com或手机13851679285联系她。她的身份证号码是320106196310023322。近期，她出现了体重超标、呼吸短促和关节不适的症状。经医疗检查，她被诊断为肥胖症，并由主治医师张明远开具了立普妥药物。沈雅琴年收入为1088870.90元，信用评分为636分。她的交易明细包含"工资收入 中国银行 REMI13101"。 My name is Ludmiła'

In [10]:
# from presidio_anonymizer import AnonymizerEngine

# anonymizer = AnonymizerEngine()

# sample = dataset[5]
# text = sample.full_text

# results = model._predict_results(sample)

# print("original_text:")
# print(text)

# anonymized_text = anonymizer.anonymize(
#     text=text,
#     analyzer_results=results,
# )

# print("anonymized_text:")
# print(anonymized_text)


In [11]:
%%time

experiment = get_experiment_tracker()

model = HanLPMixModelWrapper(
    zh_analyzer=zh_analyzer,
    en_analyzer=en_analyzer,
    labeling_scheme="IO",
)

evaluator = SpanEvaluator(model=model, iou_threshold=0.7)

params = {"dataset_name": dataset_name, "model_name": evaluator.model.name}
params.update(evaluator.model.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)

evaluation_results = evaluator.evaluate_all(dataset, language="zh")
results = evaluator.calculate_score(evaluation_results)

experiment.log_metrics(results.to_log())
entities, confmatrix = results.to_confusion_matrix()
experiment.log_confusion_matrix(matrix=confmatrix, labels=entities)
experiment.end()

results


Running model HanLPMixModelWrapper on dataset...


/Users/jyu36/code/ic-llm/presidio/.venv/lib/python3.12/site-packages/presidio_evaluator/evaluation/base_evaluator.py:83: UserWarning: skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]
  warnings.warn("skip words not provided, using default skip words. "


Finished running model on dataset
saving experiment data to /Users/jyu36/code/ic-llm/presidio/experiment_20260323-180223.json
CPU times: user 26.4 s, sys: 8.25 s, total: 34.6 s
Wall time: 24.8 s


stats=Counter({('PERSON', 'PERSON'): 532, ('O', 'PERSON'): 297, ('GENDER', 'GENDER'): 276, ('LOCATION', 'LOCATION'): 275, ('ID', 'ID'): 257, ('PERSON', 'O'): 254, ('O', 'LOCATION'): 250, ('PHONE_NUMBER', 'PHONE_NUMBER'): 247, ('EMAIL_ADDRESS', 'EMAIL_ADDRESS'): 246, ('LOCATION', 'O'): 212, ('O', 'ORGANIZATION'): 204, ('AGE', 'AGE'): 174, ('AGE', 'O'): 165, ('O', 'AGE'): 162, ('PHONE_NUMBER', 'O'): 105, ('EMAIL_ADDRESS', 'O'): 99, ('ID', 'O'): 77, ('O', 'PHONE_NUMBER'): 75, ('O', 'DATE_TIME'): 66, ('O', 'GENDER'): 50, ('GENDER', 'O'): 44, ('ORGANIZATION', 'O'): 31, ('O', 'NRP'): 26, ('O', 'EMAIL_ADDRESS'): 24, ('PERSON', 'ORGANIZATION'): 23, ('CREDIT_CARD', 'CREDIT_CARD'): 23, ('DATE_TIME', 'DATE_TIME'): 20, ('TITLE', 'O'): 18, ('AGE', 'DATE_TIME'): 16, ('ORGANIZATION', 'ORGANIZATION'): 15, ('CREDIT_CARD', 'PHONE_NUMBER'): 15, ('NRP', 'NRP'): 11, ('ZIP_CODE', 'O'): 8, ('PERSON', 'LOCATION'): 7, ('URL', 'URL'): 6, ('LOCATION', 'ORGANIZATION'): 5, ('O', 'URL'): 5, ('CREDIT_CARD', 'O'): 5,

In [12]:
from collections import Counter

predicted_label_counts = Counter(
    tag
    for r in evaluation_results
    for tag in r.predicted_tags
    if tag != "O"
)

print(predicted_label_counts)


Counter({'PERSON': 3862, 'LOCATION': 3861, 'ORGANIZATION': 2386, 'EMAIL_ADDRESS': 991, 'AGE': 661, 'PHONE_NUMBER': 496, 'DATE_TIME': 346, 'GENDER': 326, 'NRP': 324, 'ID': 257, 'URL': 48, 'IP_ADDRESS': 32, 'CREDIT_CARD': 23, 'IBAN_CODE': 16, 'US_SSN': 15, 'US_DRIVER_LICENSE': 1, 'US_BANK_NUMBER': 1})


In [13]:
# # Plot output
plotter = Plotter(results=results, 
                  model_name = evaluator.model.name, 
                  save_as="svg",
                  beta = 2) 

# Path of the directory to save the plots
output_folder = Path(Path.cwd(), "plotter_output")
plotter.plot_scores(output_folder=output_folder)

In [14]:
ModelError.most_common_fp_tokens(results.model_errors)


Most common false positive tokens:
[('的', 41),
 ('中国 工商 银行', 27),
 ('邮箱', 16),
 ('女性 话剧 导演 。 她 居住 在 江苏省 南京市 鼓楼区 中山 北路 803号 ， 可 通过 邮箱 ivan peng 6069 163.com 或 '
  '电话 13812345678 联系 。 她 的 身份证 号码 是 320106198805248620 。 最近 ， 她 出现 了 频繁 口渴 、 '
  '多尿 和 视力 模糊 的 症状 。 经 医疗 检查 ， 被 诊断 为 糖尿病 。 她 的 主治 医生 是 张伟明 医生 。 杜雨 桐 目前 正在 服用 '
  '布洛芬 作为 药物 治疗 方案 的 一部分 。 财务 方面 ， 她 的 年收入 为 765 902.99 元 ， 信用 评分 为 524分 。 与 她 '
  '相关 的 交易 详情 包括 中国 工商 银行 stl22091',
  12),
 ('男性 皮肤科 医生 。 他 居住 在 黑龙江省 哈尔滨市 南岗区 学府路 1425号 ， 可 通过 邮箱 shaomingyuan 163.com 或 '
  '手机 13945672318 联系 。 他 的 身份证 号码 是 230103198601153217 。 近期 他 出现 了 呼吸 困难 、 咳嗽 '
  '和 胸闷 等 症状 。 经谭 淑华 医生 诊断 ， 确诊 为 哮喘 并 开具 了 青霉素 治疗 。 此外 ， 邵明远 的 信用 评分 为 734分 ， '
  '月 收入 为 89654 32 元 。 交易 记录 显示 一 笔 央行 内部 资金 划转 业务',
  12),
 ('女性 教师 ， 现居 住 在 河北省 石家庄市 裕华区 槐安东路 123号 。 她 的 身份证 号码 为 130105199801012345 ， 可 '
  '通过 邮箱 mali_teacher 163.com 或 手机 13812345678 联系 。 近期 她 出现 了 不明 肿块 、 持续 疲劳 和 '
  '体重 下降 等 症状 。 经诊 断确 诊为 癌症 。 她 的 主治 医生 张建国 为 其 开具 了 阿司匹林 作为 治疗 方案 。 财务 方面 ， '
  "马丽年 收入 为 8

[('的', 41),
 ('中国 工商 银行', 27),
 ('邮箱', 16),
 ('女性 话剧 导演 。 她 居住 在 江苏省 南京市 鼓楼区 中山 北路 803号 ， 可 通过 邮箱 ivan peng 6069 163.com 或 电话 13812345678 联系 。 她 的 身份证 号码 是 320106198805248620 。 最近 ， 她 出现 了 频繁 口渴 、 多尿 和 视力 模糊 的 症状 。 经 医疗 检查 ， 被 诊断 为 糖尿病 。 她 的 主治 医生 是 张伟明 医生 。 杜雨 桐 目前 正在 服用 布洛芬 作为 药物 治疗 方案 的 一部分 。 财务 方面 ， 她 的 年收入 为 765 902.99 元 ， 信用 评分 为 524分 。 与 她 相关 的 交易 详情 包括 中国 工商 银行 stl22091',
  12),
 ('男性 皮肤科 医生 。 他 居住 在 黑龙江省 哈尔滨市 南岗区 学府路 1425号 ， 可 通过 邮箱 shaomingyuan 163.com 或 手机 13945672318 联系 。 他 的 身份证 号码 是 230103198601153217 。 近期 他 出现 了 呼吸 困难 、 咳嗽 和 胸闷 等 症状 。 经谭 淑华 医生 诊断 ， 确诊 为 哮喘 并 开具 了 青霉素 治疗 。 此外 ， 邵明远 的 信用 评分 为 734分 ， 月 收入 为 89654 32 元 。 交易 记录 显示 一 笔 央行 内部 资金 划转 业务',
  12),
 ("女性 教师 ， 现居 住 在 河北省 石家庄市 裕华区 槐安东路 123号 。 她 的 身份证 号码 为 130105199801012345 ， 可 通过 邮箱 mali_teacher 163.com 或 手机 13812345678 联系 。 近期 她 出现 了 不明 肿块 、 持续 疲劳 和 体重 下降 等 症状 。 经诊 断确 诊为 癌症 。 她 的 主治 医生 张建国 为 其 开具 了 阿司匹林 作为 治疗 方案 。 财务 方面 ， 马丽年 收入 为 861 959.07 元 ， 信用 评分 为 531分 。 标注 为 '央 行 内部 资金 划 转' 的 交易 记录 可能 与其 医疗 或 个人 需求 相关 的 内部 财务

In [15]:
fps_df = ModelError.get_fps_dataframe(results.model_errors)
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)


,full_text,token,annotation,prediction
0,她 的 电子 邮箱 是 moyaq in @ 163.com,她 的 电子 邮箱 是 moyaq 163.com,EMAIL_ADDRESS,O
1,4273468222888969924,4273468222888969924,DATE_TIME,O
2,张明远 沈雅 琴年,张明远 沈雅 琴年,PERSON,O
3,the age of 45,45,DATE_TIME,O
4,Sophie Lang \n,sophie lang \n,ORGANIZATION,O
5,3482,3482,DATE_TIME,O
6,Franscini 71,franscini 71,PERSON,O
7,Tujetsch,tujetsch,ORGANIZATION,O
8,Billing address,billing,O,LOCATION
9,崔明远,崔明远,LOCATION,O


In [16]:
ModelError.most_common_fn_tokens(results.model_errors, n=15)


Most common false negative tokens:
[('女性', 23),
 ('男性', 21),
 ('13812345678', 21),
 ('13857123456', 12),
 ('13945672318', 12),
 ('13945671234', 9),
 ('58', 7),
 ('66', 6),
 ('张建国', 6),
 ('69', 6),
 ('62', 6),
 ('shenyutong 163.com', 5),
 ('46', 5),
 ('mrs', 5),
 ('19', 5)]
---------------
Example sentence with each FN token:
	- 女性 (`女性` annotated as O)
	- 男性 (`男性` annotated as O)
	- 13812345678 (`13812345678` annotated as O)
	- 13857123456 (`13857123456` annotated as O)
	- 13945672318 (`13945672318` annotated as O)
	- 13945671234 (`13945671234` annotated as O)
	- 58 (`58` annotated as O)
	- 66 (`66` annotated as O)
	- 张建国 (`张建国` annotated as O)
	- 69 (`69` annotated as O)
	- 62 (`62` annotated as O)
	- shenyutong @ 163.com (`shenyutong 163.com` annotated as O)
	- 46 (`46` annotated as O)
	- Mrs . (`mrs` annotated as O)
	- 19 (`19` annotated as O)


[('女性', 23),
 ('男性', 21),
 ('13812345678', 21),
 ('13857123456', 12),
 ('13945672318', 12),
 ('13945671234', 9),
 ('58', 7),
 ('66', 6),
 ('张建国', 6),
 ('69', 6),
 ('62', 6),
 ('shenyutong 163.com', 5),
 ('46', 5),
 ('mrs', 5),
 ('19', 5)]

In [17]:
fns_df = ModelError.get_fns_dataframe(results.model_errors)
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)


,full_text,token,annotation,prediction
0,moyaq in @ 163.com,moyaq 163.com,O,EMAIL_ADDRESS
1,4273468222888969924,4273468222888969924,O,CREDIT_CARD
2,张明远 沈雅 琴年 Ludmiła,张明远 沈雅 琴年 ludmiła,O,PERSON
3,45,45,O,AGE
4,Sophie Lang,sophie lang,O,PERSON
5,3482 Via Franscini 71 Suite 838 Tujetsch nan,3482 franscini 71 838 tujetsch nan,O,LOCATION
6,7188,7188,O,ZIP_CODE
7,崔明远,崔明远,O,PERSON
8,27,27,O,AGE
9,辽宁省 沈阳市 和平区 青年 大街 287号,辽宁省 沈阳市 和平区 青年 大街 287号,O,LOCATION
